# Prepare Data For Pilot Study
The pilot study is examining the ByT5 parser's performance on natural language with self-repair. An error analysis based on the results will illustrate the limitations of the current version of the parser, shedding the light for future adjustments. 

First check the data completeness; then 

In [9]:
import numpy as np
import pandas as pd 
import seaborn

## 1. Formality Check

In [3]:
import os
print(os.getcwd())

/Users/hongxuzhou/Documents/GitHub/lct_master_project/colloquium_prep


In [6]:
valid_count = 0
invalid_count = 0

with open("../data/pmb-5.1.0/split/en/train/gold.sbn", "r", encoding="utf-8") as f: 
    # Split the files by two line breakers 
    blocks = f.read().strip().split("\n\n")

    for block in blocks:
        lines = block.split("\n")
        if len(lines) == 3:
            valid_count += 1
        else:
            # Capture the corrupted samples
            invalid_count += 1
            print(f"Malformed entry found: {line[0]}")
print(f"Valid samples: {valid_count}; Invalid samples: {invalid_count}")

Valid samples: 9552; Invalid samples: 0


## 2. Length Selection

I will select the samples with more than 8 words for repair-injection. 

To facilitate the following tasks, I use `Pandas` and save the data in a DF. 

In [ ]:
# Parse the data to lists
data = []
with open("../data/pmb-5.1.0/split/en/train/gold.sbn", "r", encoding="utf-8") as f: 
    block = f.read().strip("\n\n")
    for block in blocks:
        lines = block.split("\n")
        data.append({id: lines[0],
        "nl": lines[1],
        "mr": lines[2]})

# Set up the dataframe
df = pd.DataFrame(data)

# Filter the data with word len > 8 and vectorisation 
# I use space splitting for word counting for the sake of convenience 
df_filtered = df[df["nl"].str.split().str.len() > 8] # One known issue of this method is it cannot detect the fixed expressions and idioms 

# Define the names of the coming fields 
new_columns = [
    "repair_head_nl",
    "repair_head_sbn",
    "repair_head_smatch",
    "repair_mid_nl",
    "repair_mid_sbn",
    "repair_mid_smatch",
    "repair_tail.nl",
    "repair_tail_sbn",
    "repair_tail_smatch",
    "repair_interrug_nl",
    "repair_interrug_sbn"
    "repair_interrug_smatch"

]

for col in new_columns:
    if col not in df_filtered.columns:
        df_filtered[col] = ""

# Save as TSV file 
df_filtered.to_csv("pmb_pilot_data.tsv", sep="\t", index=False, encoding="utf-8")

print(f"the shape of the saved tsv is: {df_filtered.shape}")


the shape of the saved tsv is: (865, 11)


In [13]:
print(df_filtered.columns)

Index([<built-in function id>,                   'nl',                   'mr',
             'repair_head_nl',      'repair_head_sbn',        'repair_mid_nl',
             'repair_mid_sbn',       'repair_tail.nl',      'repair_tail_sbn',
         'repair_interrug_nl',  'repair_interrug_sbn'],
      dtype='object')
